[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/klar74/WS2026_lecture/blob/main/Vorlesung_26/VL26_RAG_Demo.ipynb)

# VL26 Notebook 3: RAG – Dokumente suchen und Antworten vorbereiten

**RAG** steht für *Retrieval-Augmented Generation*: Zuerst werden passende Dokumente gesucht, dann erhält ein Sprachmodell diese Dokumente zusammen mit der Frage als Kontext für seine Antwort.

Beispiel: Auf die Frage „Wie viele Urlaubstage habe ich?“ suchen wir in einem fiktiven Firmenhandbuch nach einer Urlaubsregelung. Dieses Notebook zeigt die Suche und erstellt den Text für ein Sprachmodell. **Es ruft kein Sprachmodell zur Antwortgenerierung auf** und benötigt keinen API-Key.

In [ ]:
# Installation (nur einmal ausführen)
# !pip install sentence-transformers numpy scikit-learn matplotlib

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

## 1. Wie funktioniert RAG?

Ein großes Sprachmodell (**LLM**) kennt nicht automatisch die aktuellen internen Regeln einer Firma. Ohne passende Informationen kann es eine falsche Antwort geben.

RAG ergänzt die Frage um Informationen aus Dokumenten:
1. **Retrieval – Suche:** Dokumente zur Frage auswählen.
2. **Augmentation – Ergänzung:** Die gefundenen Dokumente und die Frage in einen Prompt (eine Eingabe für das Sprachmodell) schreiben.
3. **Generation – Antwort:** Ein Sprachmodell verwendet diesen Prompt, um eine Antwort zu formulieren. Diesen Schritt führen wir hier nicht aus.

Die Suche kann unpassende Dokumente liefern, und auch ein Sprachmodell kann die Quellen falsch verwenden. RAG garantiert daher keine richtige Antwort.

## 2. Schritt 1: Dokumente vorbereiten

Wir verwenden fünf kurze, frei erfundene Dokumente mit Regeln zu Urlaub, Arbeitszeit und weiteren Themen. Ihre Inhalte sind keine realen Firmenregeln.

In [ ]:
# Beispiel-Dokumente (Firmen-Handbuch)
dokumente = [
    {
        "id": "DOC001",
        "titel": "Urlaubsregelung",
        "text": "Mitarbeiter haben Anspruch auf 30 Tage Urlaub pro Jahr. Urlaub muss mindestens 2 Wochen im Voraus beantragt werden."
    },
    {
        "id": "DOC002", 
        "titel": "Arbeitszeiten",
        "text": "Die reguläre Arbeitszeit beträgt 40 Stunden pro Woche, Montag bis Freitag von 9:00 bis 17:00 Uhr. Gleitzeit ist möglich zwischen 7:00 und 19:00 Uhr."
    },
    {
        "id": "DOC003",
        "titel": "Homeoffice-Regelung", 
        "text": "Mitarbeiter können bis zu 3 Tage pro Woche im Homeoffice arbeiten. Dienstags ist Präsenzpflicht für Team-Meetings."
    },
    {
        "id": "DOC004",
        "titel": "Krankheit",
        "text": "Bei Krankheit muss der Vorgesetzte am ersten Tag telefonisch informiert werden. Ab dem 3. Tag ist ein ärztliches Attest erforderlich."
    },
    {
        "id": "DOC005",
        "titel": "Weiterbildung",
        "text": "Jeder Mitarbeiter hat ein jährliches Weiterbildungsbudget von 2000 Euro. Schulungen müssen mit der Personalabteilung abgestimmt werden."
    }
]

print(f"Anzahl Dokumente: {len(dokumente)}")
for doc in dokumente:
    print(f"  - {doc['id']}: {doc['titel']}")

## 3. Schritt 2: Dokumente in Vektoren umwandeln

Ein **Embedding** ist ein Vektor, also eine Liste von Zahlen. Das Modell **Sentence-Transformers** erzeugt für jeden Dokumententext einen solchen Vektor. Texte mit ähnlichem Inhalt können ähnliche Vektoren erhalten.

Die Vektoren bleiben hier als NumPy-Array im Arbeitsspeicher; wir verwenden keine Vektordatenbank. Das mehrsprachige Modell wird bei Bedarf aus dem Internet heruntergeladen. Auch beim späteren Laden kann die Bibliothek eine Internetverbindung benötigen.

In [ ]:
# Mehrsprachiges Sentence-Transformer-Modell laden
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')
print("✓ Embedding-Modell geladen!")

# Alle Dokumente in Vektoren umwandeln
doc_texte = [doc["text"] for doc in dokumente]
doc_embeddings = model.encode(doc_texte)

print("\nEmbeddings erstellt:")
print(f"  Form des Arrays: {doc_embeddings.shape}")
print(f"  {doc_embeddings.shape[0]} Dokumente, je {doc_embeddings.shape[1]} Zahlen")

## 4. Schritt 3: Dokumente zur Frage suchen

Auch die Frage wird in einen Vektor umgewandelt. Die **Kosinus-Ähnlichkeit** vergleicht seine Richtung mit jedem Dokumentvektor. Die Funktion sortiert die Dokumente nach diesem Wert und gibt die `top_k` höchsten Treffer zurück.

Die Suche kann ähnliche Formulierungen finden, auch wenn Wörter nicht genau übereinstimmen. Ein hoher Wert ist aber weder eine Wahrscheinlichkeit noch ein Beweis, dass ein Dokument die Frage beantwortet.

In [ ]:
def suche_dokumente(frage, top_k=2):
    """
    Findet die Dokumente mit der höchsten Vektorähnlichkeit zur Frage.
    
    Args:
        frage: Frage als Zeichenkette
        top_k: Anzahl der zurückzugebenden Dokumente
    
    Returns:
        Liste der gefundenen Dokumente mit Ähnlichkeitswerten
    """
    if not isinstance(top_k, int) or isinstance(top_k, bool) or not 1 <= top_k <= len(dokumente):
        raise ValueError(f"top_k muss zwischen 1 und {len(dokumente)} liegen.")

    # Frage in Vektor umwandeln
    frage_embedding = model.encode([frage])
    
    # Ähnlichkeit zu allen Dokumenten berechnen
    similarities = cosine_similarity(frage_embedding, doc_embeddings)[0]
    
    # Top K Indizes finden
    top_indices = np.argsort(similarities)[-top_k:][::-1]
    
    # Ergebnisse zusammenstellen
    results = []
    for idx in top_indices:
        results.append({
            "dokument": dokumente[idx],
            "aehnlichkeit": similarities[idx]
        })
    
    return results

# Test: Verschiedene Fragen
fragen = [
    "Wie viele Urlaubstage habe ich?",
    "Kann ich von zu Hause arbeiten?",
    "Was muss ich bei Krankheit tun?",
    "Wie viel Geld gibt es für Fortbildung?"
]

for frage in fragen:
    print(f"\n{'='*60}")
    print(f"Frage: {frage}")
    print(f"{'='*60}")
    
    results = suche_dokumente(frage, top_k=2)
    
    for i, result in enumerate(results, 1):
        doc = result["dokument"]
        aehnlichkeit = result["aehnlichkeit"]
        print(f"\n{i}. {doc['titel']} (Kosinus-Ähnlichkeit: {aehnlichkeit:.3f})")
        print(f"   {doc['text']}")

**Zum Prüfen:** Steht bei „von zu Hause arbeiten“ die Homeoffice-Regelung weit oben? Die Suche vergleicht die **Dokumenttexte**, nicht nur ihre Titel. Auch unpassende Treffer können in der Liste stehen.

## 5. Schritt 4: RAG-Prompt konstruieren

Wir fügen die gefundenen Dokumente und die Frage zu einem Prompt zusammen. Hier wird dieser Prompt nur angezeigt – er wird nicht an ein Sprachmodell gesendet. Die Anweisung, nur anhand der Dokumente zu antworten, kann ein Sprachmodell unterstützen, aber nicht garantieren, dass es sich daran hält.

In [ ]:
def erstelle_rag_prompt(frage, top_k=2):
    """
    Erstellt einen RAG-Prompt mit Kontext-Dokumenten.
    """
    # Relevante Dokumente finden
    results = suche_dokumente(frage, top_k)
    
    # Kontext zusammenbauen
    kontext = "\n\n".join([
        f"Dokument {i+1} ({r['dokument']['titel']}):\n{r['dokument']['text']}"
        for i, r in enumerate(results)
    ])
    
    # Prompt konstruieren
    prompt = f"""Du bist ein hilfreicher Assistent, der Fragen basierend auf Firmen-Dokumenten beantwortet.

WICHTIG: Beantworte die Frage NUR basierend auf den folgenden Dokumenten. 
Wenn die Antwort nicht in den Dokumenten steht, sage das deutlich.

KONTEXT:
{kontext}

FRAGE: {frage}

ANTWORT:"""
    
    return prompt, results

# Beispiel
frage = "Wie viele Tage Urlaub stehen mir zu?"
prompt, docs = erstelle_rag_prompt(frage)

print("RAG-Prompt:")
print("="*60)
print(prompt)
print("="*60)
print(f"\nVerwendete Dokumente:")
for doc in docs:
    print(f"  - {doc['dokument']['titel']} (Kosinus-Ähnlichkeit: {doc['aehnlichkeit']:.3f})")

## 6. Prüfen: Findet die Suche das passende Dokument?

Für drei Fragen legen wir von Hand fest, welches Dokument die Antwort enthält. Wir vergleichen diese Erwartung mit dem ersten Suchtreffer. Zusätzlich stellen wir eine Frage zum WLAN-Passwort: Keines der fünf Dokumente enthält diese Information.

Die Suche liefert trotzdem immer einen Treffer, weil `top_k=1` genau ein Dokument anfordert. Ein Treffer bedeutet also nicht automatisch, dass die Antwort vorhanden ist. Die Erwartungen unten sind von Hand vorgegeben; das Modell erzeugt keine Antworten.

In [ ]:
# Erwartete Dokumente von Hand festlegen; None bedeutet: keine Antwort vorhanden.
test_faelle = [
    ("Muss ich bei Krankheit ein Attest vorlegen?", "DOC004"),
    ("Wann muss ich ins Büro kommen?", "DOC003"),
    ("Wie hoch ist mein Weiterbildungsbudget?", "DOC005"),
    ("Wie lautet das WLAN-Passwort?", None),
]

for frage, erwartet in test_faelle:
    treffer = suche_dokumente(frage, top_k=1)[0]
    doc = treffer['dokument']
    print(f"\nFrage: {frage}")
    print(f"  Erster Treffer: {doc['id']} – {doc['titel']} "
          f"(Kosinus-Ähnlichkeit: {treffer['aehnlichkeit']:.3f})")
    if erwartet is None:
        print("  Erwartung: Kein Dokument enthält die Antwort.")
    else:
        print(f"  Erwartetes Dokument: {erwartet} | Treffer stimmt: {doc['id'] == erwartet}")

## 7. Ähnlichkeiten visualisieren

Für die Urlaubsfrage zeigen wir die Kosinus-Ähnlichkeit zu allen fünf Dokumenten. Ein längerer Balken bedeutet einen höheren Ähnlichkeitswert – nicht, dass die Antwort mit dieser Wahrscheinlichkeit richtig ist.

In [ ]:
import matplotlib.pyplot as plt

grafik_frage = "Wie viele Tage Urlaub stehen mir zu?"
results = suche_dokumente(grafik_frage, top_k=len(dokumente))
aehnlichkeiten = [r['aehnlichkeit'] for r in results]
titel = [r['dokument']['titel'] for r in results]

plt.figure(figsize=(8, 3))
plt.barh(titel, aehnlichkeiten, color='steelblue')
plt.gca().invert_yaxis()
plt.xlim(-1, 1)
plt.xlabel('Kosinus-Ähnlichkeit')
plt.title(grafik_frage)
plt.tight_layout()
plt.show()

## 🎯 Takeaway

- Wir haben Dokumente und Fragen in Vektoren umgewandelt und anhand ihrer Kosinus-Ähnlichkeit gesucht.
- Die Suche gibt auch dann Treffer zurück, wenn kein Dokument eine Antwort enthält.
- Ein RAG-Prompt ergänzt die Frage um gefundene Texte. Ob diese passen und aktuell sind, muss geprüft werden.
- Für eine tatsächliche Antwortgenerierung müsste ein Sprachmodell aufgerufen werden. Das geschieht in diesem Notebook nicht.

---

## 🎓 Ende der VL26 Notebooks

**Was wir gelernt haben:**
1. **Word2Vec**: Feste Wortvektoren aus Wortumgebungen lernen
2. **BERT**: Kontextabhängige Embeddings (Bank = Geldinstitut vs. Sitzmöbel)
3. **RAG**: Dokumente suchen und als Kontext für ein Sprachmodell bereitstellen

**Kein API-Key nötig:** Das Embedding-Modell wird bei Bedarf aus dem Internet geladen. Nach dem Laden läuft die Suche in der Notebook-Umgebung; ein LLM wird in diesem Notebook nicht gestartet.